# 🏨 Hotel QA-ассистент на LangGraph + GigaChat

**Цель:** Построить QA-ассистента отеля на LangGraph с использованием GigaChat API,  
прогнать на эталонном наборе из 20 вопросов и оценить качество ответов.

**Модель:** GigaChat (Сбер) — `GIGACHAT_API_PERS`  

**Структура:**
1. Установка зависимостей и импорты
2. Клиент GigaChat (OAuth2 + авторефреш токена)
3. Данные: буклет отеля и эталонные Q&A пары
4. Обоснование метрик оценки
5. Вспомогательные функции (ANLS + LLM-as-a-Judge)
6. Граф-ассистент с классификатором out-of-scope (бонус)
7. Граф-пайплайн оценки
8. Запуск и результаты
9. Детальный анализ и сравнение двух метрик (бонус)

## 1. Установка зависимостей и импорты

In [29]:
# !pip install langgraph requests -q

import os
import re
import time
import uuid
import operator
from difflib import SequenceMatcher
from typing import TypedDict, List, Optional, Annotated

import requests
import urllib3

from langgraph.graph import StateGraph, END, START

# GigaChat использует сертификат Минцифры/Сбера.
# В production добавьте CA-bundle; здесь отключаем предупреждения для простоты.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print('✅ Импорты успешны')

✅ Импорты успешны


## 2. Клиент GigaChat (OAuth2 + авторефреш токена)

GigaChat использует двухшаговую авторизацию:
1. `POST /oauth` с `Authorization: Basic <auth_key>` → `access_token` (живёт ~30 мин)
2. `POST /chat/completions` с `Authorization: Bearer <access_token>`

`GigaChatClient` кэширует токен и обновляет его автоматически при истечении.

In [30]:
class GigaChatClient:
    """
    Минималистичный клиент GigaChat API с авторефрешем OAuth2-токена.

    Параметры:
        auth_key  -- Authorization key из настроек проекта (base64-строка)
        scope     -- GIGACHAT_API_PERS | GIGACHAT_API_B2B | GIGACHAT_API_CORP
        model     -- GigaChat | GigaChat-Pro | GigaChat-Max
    """

    TOKEN_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth'
    CHAT_URL  = 'https://gigachat.devices.sberbank.ru/api/v1/chat/completions'

    def __init__(self, auth_key: str, scope: str = 'GIGACHAT_API_PERS',
                 model: str = 'GigaChat'):
        self.auth_key    = auth_key
        self.scope       = scope
        self.model       = model
        self._token:      Optional[str] = None
        self._expires_at: float         = 0.0

    # ---------- авторизация ----------

    def _refresh_token(self) -> None:
        """Получает новый access_token."""
        resp = requests.post(
            self.TOKEN_URL,
            headers={
                'Content-Type':  'application/x-www-form-urlencoded',
                'Accept':        'application/json',
                'RqUID':         str(uuid.uuid4()),
                'Authorization': f'Basic {self.auth_key}',
            },
            data={'scope': self.scope},
            verify=False,
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()
        self._token = data['access_token']
        # expires_at приходит в миллисекундах
        expires_ms       = data.get('expires_at', (time.time() + 1800) * 1000)
        self._expires_at = expires_ms / 1000.0
        print(f'  [GigaChat] Токен получен, действует до '
              f'{time.strftime("%H:%M:%S", time.localtime(self._expires_at))}')

    def _get_token(self) -> str:
        if not self._token or time.time() >= self._expires_at - 60:
            self._refresh_token()
        return self._token

    # ---------- чат ----------

    def chat(self, messages: List[dict], system: Optional[str] = None,
             max_tokens: int = 600, temperature: float = 0.3) -> str:
        """
        Вызывает /chat/completions и возвращает текст ответа.

        Args:
            messages    -- список {role, content} (без системного)
            system      -- системный промпт (добавляется первым с role=system)
            max_tokens  -- ограничение на длину ответа
            temperature -- температура генерации (0 = детерминировано)
        """
        msgs = []
        if system:
            msgs.append({'role': 'system', 'content': system})
        msgs.extend(messages)

        resp = requests.post(
            self.CHAT_URL,
            headers={
                'Accept':        'application/json',
                'Content-Type':  'application/json',
                'Authorization': f'Bearer {self._get_token()}',
            },
            json={
                'model':       self.model,
                'messages':    msgs,
                'max_tokens':  max_tokens,
                'temperature': temperature,
            },
            verify=False,
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content'].strip()


# ── Инициализация ─────────────────────────────────────────────
# Вставьте Authorization key (base64) из настроек проекта GigaChat,
# или задайте переменную окружения:
#   export GIGACHAT_AUTH_KEY="ваш_ключ"

GIGACHAT_AUTH_KEY = os.environ.get('GIGACHAT_AUTH_KEY', 'MDE5ZGI1NzEtYTc4MC03ZTA2LTkyM2EtZDI2YjNkOWFhMmUzOjBiMjdhNzQ4LTlkMzgtNDQ1Ni1hMWFjLWYwZDNlNzVlMzFmYQ==')
# client_id проекта: 019db571-a780-7e06-923a-d26b3d9aa2e3

gc = GigaChatClient(
    auth_key = GIGACHAT_AUTH_KEY,
    scope    = 'GIGACHAT_API_PERS',
    model    = 'GigaChat',           # GigaChat-Pro даёт лучшее качество
)

# Проверка соединения
try:
    test = gc.chat([{'role': 'user', 'content': 'Скажи только: OK'}], max_tokens=5)
    print(f'✅ GigaChat подключён. Ответ на тест: {test}')
except Exception as e:
    print(f'❌ Ошибка подключения к GigaChat: {e}')
    print('   Проверьте GIGACHAT_AUTH_KEY и доступность серверов Сбера.')

  [GigaChat] Токен получен, действует до 18:29:00
✅ GigaChat подключён. Ответ на тест: OK


## 3. Данные: буклет отеля и Q&A пары

In [31]:
# ================================================================
# hotel_booklet.docx — единственный источник знаний ассистента.
# Загружается один раз и передаётся как системный контекст.
# Весь документ (~3K токенов) помещается в окно GigaChat.
# ================================================================

HOTEL_BOOKLET = """
ДОБРО ПОЖАЛОВАТЬ В THE GRAND HARBOR HOTEL
Ваш дом вдали от дома

Уважаемый гость,
Добро пожаловать в The Grand Harbor Hotel! Мы рады принять вас и сделаем всё,
чтобы ваше пребывание было комфортным, запоминающимся и приятным.
Если потребуется помощь, наша служба ресепшен работает круглосуточно — добавочный 0.

СЛУЖБА РЕСЕПШЕН
Часы работы: круглосуточно. Телефон: добавочный 0.
Время заезда: 15:00. Время выезда: 11:00.
Ранний заезд и поздний выезд:
- Ранний заезд (с 13:00): доплата $25, при наличии свободных номеров
- Поздний выезд (до 13:00): доплата $25
- Поздний выезд (до 15:00): доплата $50
Позднее прибытие: если планируете приехать после 23:00, предупредите заранее.

ИНФОРМАЦИЯ О НОМЕРАХ
Стандартный номер: кондиционер, ТВ с кабельными каналами, бесплатный Wi-Fi,
мини-холодильник, кофемашина, утюг, фен, принадлежности для ванной, ежедневная уборка.
Номер Deluxe: всё то же + микроволновая печь и зона отдыха, расширенный набор принадлежностей.
Сьют: отдельная гостиная, мини-кухня, обеденная зона, премиальные принадлежности.
Вместимость: стандартные — до 2 взрослых; Deluxe — до 4 гостей; сьюты — до 6 гостей.
Дети до 12 лет — бесплатно при размещении на имеющихся спальных местах с родителями.

ПИТАНИЕ
Завтрак: доступен на отдельных тарифах. Часы: 6:30–10:00. Ресторан, 1-й этаж. Цена: $15/чел.
Рум-сервис: 6:30–23:00 ежедневно; ночное меню 23:00–1:00 только по выходным.
Доставка: $3.50 за заказ. Телефон: добавочный 7.

УДОБСТВА ОТЕЛЯ
Бассейн: 6:00–22:00 ежедневно (при подходящей погоде), открытый, во внутреннем дворе.
Дети до 14 лет — только со взрослым. Полотенца выдаются на ресепшен.
Фитнес-центр: круглосуточно, доступ по ключу от номера, 2-й этаж.
Оборудование: кардиотренажёры, свободные веса, силовое оборудование.
Бизнес-центр: круглосуточно, главный лобби. Услуги: компьютер, печать, копирование, факс.

ИНТЕРНЕТ
Бесплатный Wi-Fi: сеть HotelGuest, пароль — при заселении.
Премиум Wi-Fi: $9.99/день (повышенная скорость).

ПАРКОВКА И ТРАНСПОРТ
Самостоятельная парковка: $12/ночь. С парковщиком: $20/ночь.
Зарядка электромобиля: $5/ночь (мест ограничено). В высокий сезон парковка ограничена.
Трансфер из аэропорта: бесплатно, 5:00–23:00.
В часы пик — каждые 30 мин, в остальное время — раз в час.
Бронирование: звоните по прибытии или сообщите ресепшен заранее.

ДОПОЛНИТЕЛЬНЫЕ УСЛУГИ
Прачечная: профессиональная — возврат на следующий день (сдать до 10:00).
Самообслуживание: машины на 2-м этаже за монеты, круглосуточно.
Стиральный порошок продаётся на ресепшен.
Дополнительные спальные места: раскладные кровати $25/ночь (бронировать заранее);
детские кроватки и манежи — бесплатно по запросу. Звоните минимум за 24 часа.
Бюро находок: вещи хранятся 90 дней; без контактных данных — 30 дней, затем благотворительность.
Можем отправить вещи за ваш счёт.

ПРАВИЛА ОТЕЛЯ
Курение: все помещения — для некурящих. Курить разрешено: терраса (3-й этаж) и внутренний двор.
Штраф за курение в номере: $250.
Животные: принимаем собак и кошек, $25/ночь. На поводке в общих зонах.
Нельзя оставлять одних в номере. Лежанки и миски — по запросу. Сообщите при бронировании.
Тихие часы: 22:00–7:00. Жалобы на шум: добавочный 0.
Документы при заселении: действующий документ с фото (ВУ, паспорт, ID-карта).
Имя в документе совпадает с бронью. Кредитная карта обязательна для авторизации доп. расходов.
Иностранным гостям — паспорт обязателен. Не основной гость — нужно письменное разрешение.
Оплата: Visa, MasterCard, American Express, Discover, дебетовые карты, наличные.
Не принимаем: личные чеки и дорожные чеки.
Отмена бронирования: без штрафа — за 24 часа до заезда.
Менее 24 часов — стоимость одной ночи. Спецтарифы/групповые — условия могут отличаться.

ДОСТУПНАЯ СРЕДА
Номера ADA: расширенные дверные проёмы, душ без бортика, опущенные штанги,
доступная сантехника, визуальные и звуковые элементы. Собаки-поводыри — без доплаты.
Укажите потребности при бронировании.

БЕЗОПАСНОСТЬ
Пожар: лестница (не лифт). Медицинская помощь: добавочный 0. Полиция: 911.
Не сообщайте номер комнаты незнакомым звонящим.

ВАЖНЫЕ ТЕЛЕФОНЫ
Ресепшен: доб. 0. Рум-сервис: доб. 7. Уборка: доб. 5. Бизнес-центр: доб. 8.
Основной номер отеля: (555) 123-4567. Экстренные службы: 911. Такси: (555) 987-6543.

ВЫЕЗД
Стандартное время: 11:00.
Экспресс-выезд: счёт через ТВ или приложение, ключи оставьте в номере.
Выезд через ресепшен: ключи на стойку для итоговой сверки.
Дополнительные списания — в течение 24 часов после выезда.
"""

print(f'✅ Буклет загружен: {len(HOTEL_BOOKLET)} символов')

✅ Буклет загружен: 4428 символов


In [32]:
# ================================================================
# Эталонные Q&A пары из hotel_qa.json.
# Используются ТОЛЬКО для оценки — НЕ передаются ассистенту!
# ================================================================

QA_PAIRS = [
    {'id': 1,  'question': 'Сколько гостей можно разместить в номере?',
     'answer': 'Стандартные номера рассчитаны на 2 взрослых. В номерах категории Deluxe могут разместиться до 4 гостей. Дети до 12 лет проживают бесплатно при размещении на имеющихся спальных местах вместе с родителями. Для больших групп свяжитесь с нами, чтобы обсудить смежные номера или варианты сьютов.'},
    {'id': 2,  'question': 'Можно ли с домашними животными?',
     'answer': 'Да, мы pet-friendly отель! Принимаем собак и кошек, доплата за животное — $25 за ночь. В общих зонах животные должны быть на поводке, оставлять их одних в номере нельзя. По запросу предоставляем лежанки и миски. Пожалуйста, сообщите о животном при бронировании.'},
    {'id': 3,  'question': 'Можно ли курить в номерах?',
     'answer': 'Все наши номера — для некурящих. Места для курения оборудованы на открытой террасе на 3-м этаже и во внутреннем дворе. За курение в номере взимается штраф $250 за уборку.'},
    {'id': 4,  'question': 'Завтрак включён в стоимость?',
     'answer': "Включён ли завтрак, зависит от тарифа. Тарифы 'С завтраком' включают завтрак с 6:30 до 10:00 ежедневно. Если завтрак не входит в ваш тариф, его можно купить за $15 с человека."},
    {'id': 5,  'question': 'Какие условия отмены брони?',
     'answer': 'Стандартные брони можно отменить без штрафа не позднее чем за 24 часа до даты заезда. При отмене менее чем за 24 часа удерживается стоимость одной ночи. Для специальных тарифов и групповых броней условия могут отличаться.'},
    {'id': 6,  'question': 'Во сколько заезд и выезд?',
     'answer': 'Заезд — с 15:00, выезд — до 11:00. Ранний заезд возможен с 13:00 за доплату $25 при наличии свободных номеров. Поздний выезд до 13:00 — за $25, до 15:00 — за $50.'},
    {'id': 7,  'question': 'Есть ли парковка?',
     'answer': 'Да, доступны самостоятельная парковка и услуги паркинга. Самостоятельная парковка — $12 за ночь, паркинг с парковщиком — $20 за ночь. Также есть зарядные станции для электромобилей за $5 за ночь.'},
    {'id': 8,  'question': 'Есть ли Wi-Fi и бесплатный ли он?',
     'answer': "Да, бесплатный высокоскоростной Wi-Fi доступен по всему отелю. Сеть называется 'HotelGuest', пароль выдадут при заселении. Для деловых поездок есть премиум Wi-Fi с повышенной скоростью за $9.99 в день."},
    {'id': 9,  'question': 'Что входит в номер?',
     'answer': 'Во всех номерах есть кондиционер, плоский ТВ с кабельными каналами, мини-холодильник, кофемашина, утюг и гладильная доска, фен и набор косметики. В номерах Deluxe дополнительно есть микроволновка и зона отдыха. В сьютах — отдельная гостиная и мини-кухня.'},
    {'id': 10, 'question': 'Какие часы работы бассейна и фитнес-центра?',
     'answer': 'Открытый бассейн работает с 6:00 до 22:00 ежедневно при подходящей погоде. Фитнес-центр доступен гостям круглосуточно — для входа потребуется ключ от номера. Полотенца для бассейна выдаются на стойке регистрации. Дети до 14 лет должны находиться в зоне бассейна со взрослым.'},
    {'id': 11, 'question': 'Можно ли заселиться после полуночи?',
     'answer': 'Да, у нас круглосуточная стойка регистрации, заселение возможно в любое время. Если планируете прибыть после 23:00, лучше предупредить нас заранее.'},
    {'id': 12, 'question': 'Можно ли запросить дополнительную кровать или детскую кроватку?',
     'answer': 'Да, раскладные кровати — $25 за ночь, их количество ограничено, поэтому запрашивайте заранее. Детские кроватки предоставляются бесплатно по запросу. Также бесплатно можно получить манеж. Позвоните минимум за 24 часа до заезда.'},
    {'id': 13, 'question': 'Доступны ли номера для гостей на инвалидной коляске?',
     'answer': 'Да, у нас есть номера, отвечающие стандартам ADA: расширенные дверные проёмы, душ без бортика, опущенные штанги в шкафах и доступная сантехника. Собаки-поводыри допускаются без доплат.'},
    {'id': 14, 'question': 'Есть ли услуги прачечной?',
     'answer': 'Да, доступны и услуги прачечной, и самообслуживание. Профессиональная стирка — возврат на следующий день (вещи сданные до 10:00). Также есть стиральные и сушильные машины на 2-м этаже, работающие 24/7 за монеты. Стиральный порошок можно купить на стойке регистрации.'},
    {'id': 15, 'question': 'Какие часы работы рум-сервиса?',
     'answer': 'Рум-сервис работает с 6:30 до 23:00 ежедневно. По выходным с 23:00 до 1:00 доступно ограниченное меню. К каждому заказу добавляется доставка $3.50. Звоните по добавочному 7.'},
    {'id': 16, 'question': 'Есть ли трансфер из аэропорта?',
     'answer': 'Да, мы предоставляем бесплатный трансфер в аэропорт и из аэропорта с 5:00 до 23:00. В часы пик автобусы ходят каждые 30 минут, в остальное время — раз в час. Рекомендуем бронировать трансфер заранее.'},
    {'id': 17, 'question': 'Какие способы оплаты вы принимаете?',
     'answer': 'Принимаем все основные карты (Visa, MasterCard, American Express, Discover), дебетовые карты и наличные. Кредитная карта обязательна при заселении. Личные и дорожные чеки не принимаем.'},
    {'id': 18, 'question': 'Какие документы нужны при заселении?',
     'answer': 'Нужен действующий государственный документ с фото (водительское удостоверение, паспорт или ID). Имя должно совпадать с бронью. Для иностранных гостей обязателен паспорт. Если вы не основной гость в брони, потребуется письменное разрешение.'},
    {'id': 19, 'question': 'Как сообщить о шуме или беспокойстве?',
     'answer': 'Сразу позвоните на стойку регистрации по добавочному 0 с номерного телефона. У нас тихий час с 22:00 до 7:00, и мы серьёзно относимся к жалобам на шум.'},
    {'id': 20, 'question': 'Я забыл вещи в номере после выезда. Есть ли бюро находок?',
     'answer': 'Да, у нас есть бюро находок. Свяжитесь с нами как можно скорее. Мы храним находки 90 дней, после чего передаём на благотворительность. Если вещь найдена, мы можем отправить её вам за ваш счёт или вы можете забрать сами.'},
]

print(f'✅ Загружено {len(QA_PAIRS)} эталонных пар')

✅ Загружено 20 эталонных пар


## 4. Обоснование метрик оценки

### Основная — **LLM-as-a-Judge** (GigaChat в роли судьи)

**Почему выбран этот метод:**  
Ответы об отеле содержат конкретные факты, но их можно сформулировать по-разному.
Например, «работает с шести до десяти» и «часы завтрака: 6:30–10:00» — семантически
одно и то же, но лексически расходятся. GigaChat в роли судьи понимает смысловую
эквивалентность, тогда как строковые метрики её не видят.

| Критерий | LLM-Judge | ANLS | BLEU/ROUGE |
|---|:---:|:---:|:---:|
| Перефразирование | ✅ | ⚠️ | ❌ |
| Частичная правота | ✅ | ⚠️ | ❌ |
| Порядок слов не важен | ✅ | ⚠️ | ❌ |
| Без доп. API-вызовов | ❌ | ✅ | ✅ |
| Русский язык | ✅ | ✅ | ⚠️ |

**Минус:** каждый вопрос стоит 3 API-вызова вместо 2 (ответ + оценка). При 20 вопросах — приемлемо.

### Бонусная — **ANLS** (Average Normalized Levenshtein Similarity)

Стандарт для Document QA (DocVQA). Быстрая, воспроизводимая baseline-метрика без API.  
**Ограничение:** чувствительна к длине строки и порядку слов — параграфные ответы занижаются.

## 5. Вспомогательные функции

In [33]:
def compute_anls(prediction: str, ground_truth: str, threshold: float = 0.5) -> float:
    """
    Average Normalized Levenshtein Similarity.

    NLS = SequenceMatcher.ratio() ≈ 1 - нормализованное редакционное расстояние.
    Если NLS < threshold — возвращаем 0 (штраф за совсем плохие ответы).
    Диапазон: [0, 1]
    """
    pred = prediction.lower().strip()
    gt   = ground_truth.lower().strip()
    if not pred and not gt:
        return 1.0
    if not pred or not gt:
        return 0.0
    nls = SequenceMatcher(None, pred, gt).ratio()
    return nls if nls >= threshold else 0.0


def llm_judge_score(question: str, prediction: str, ground_truth: str) -> float:
    """
    LLM-as-a-Judge: GigaChat оценивает ответ ассистента относительно эталона.
    Возвращает оценку в диапазоне [0, 1] (нормализованная из 0–10).
    """
    prompt = f"""Оцени качество ответа ассистента отеля по шкале от 0 до 10.

Вопрос гостя: {question}

Эталонный ответ (правильный): {ground_truth}

Ответ ассистента (оцениваем): {prediction}

Критерии:
- 9–10: Полный и точный ответ, семантически эквивалентен эталону
- 7–8: Содержит всю ключевую информацию, незначительные расхождения
- 5–6: Частично верный, есть основная информация, но упущены детали
- 3–4: Неполный или с неточностями
- 1–2: В основном неверный
- 0: Полностью неверный или отказ отвечать

Ответь ТОЛЬКО одним числом от 0 до 10."""

    try:
        raw = gc.chat(
            [{'role': 'user', 'content': prompt}],
            max_tokens=5,
            temperature=0.0,
        )
        match = re.search(r'\d+(?:\.\d+)?', raw)
        score = float(match.group()) if match else 0.0
        return max(0.0, min(10.0, score)) / 10.0
    except Exception as e:
        print(f'    [judge error] {e}')
        return 0.0


# Smoke-test
print('ANLS smoke-test:')
print(f'  идентичные:          {compute_anls("бассейн 6:00–22:00", "бассейн 6:00–22:00"):.3f}')
print(f'  перефразированные:   {compute_anls("работает до 22 вечера", "бассейн 6:00–22:00"):.3f}')
print(f'  пустой предикт:      {compute_anls("", "нет ответа"):.3f}')

ANLS smoke-test:
  идентичные:          1.000
  перефразированные:   0.000
  пустой предикт:      0.000


## 6. Граф-ассистент (LangGraph + GigaChat)

```
START
  │
  ▼
[classify_question]  ← GigaChat: вопрос про отель?
  │
  ├── «hotel»        ──► [answer_question]       → END
  │                        (буклет = system prompt)
  └── «out_of_scope» ──► [out_of_scope_response] → END
```

In [34]:
# ── Состояние ─────────────────────────────────────────────────
class AssistantState(TypedDict):
    question:         str
    is_hotel_related: Optional[bool]
    answer:           str


# ── Узел 1: классификатор (бонус — out-of-scope) ──────────────
def classify_question(state: AssistantState) -> dict:
    """Определяет, относится ли вопрос к теме гостиницы."""
    raw = gc.chat(
        [{'role': 'user', 'content': (
            'Определи, относится ли следующий вопрос к теме гостиницы '
            '(заселение, выезд, номера, услуги, правила, питание, парковка, '
            'Wi-Fi, бассейн, бронирование, оплата, трансфер и т.п.).\n\n'
            f'Вопрос: {state["question"]}\n\n'
            'Ответь ТОЛЬКО одним словом: да или нет.'
        )}],
        max_tokens=5,
        temperature=0.0,
    )
    return {'is_hotel_related': 'да' in raw.lower()}


# ── Узел 2: ответ по буклету ───────────────────────────────────
def answer_question(state: AssistantState) -> dict:
    """
    Отвечает на вопрос гостя, используя ТОЛЬКО буклет отеля.
    Буклет передаётся как system-контекст — единственный источник знаний.
    """
    answer = gc.chat(
        [{'role': 'user', 'content': state['question']}],
        system=(
            'Ты — вежливый ассистент отеля The Grand Harbor Hotel. '
            'Отвечай на вопросы гостей кратко и точно, опираясь ТОЛЬКО '
            'на информацию из буклета ниже. Не придумывай данные. '
            'Если информации нет в буклете, честно скажи об этом.\n\n'
            f'БУКЛЕТ ОТЕЛЯ:\n{HOTEL_BOOKLET}'
        ),
        max_tokens=600,
        temperature=0.3,
    )
    return {'answer': answer}


# ── Узел 3: out-of-scope ───────────────────────────────────────
def out_of_scope_response(state: AssistantState) -> dict:
    return {'answer': (
        'Извините, я специализируюсь только на вопросах, связанных с '
        'The Grand Harbor Hotel — заселение, услуги, удобства, правила и т.п. '
        'По другим темам, пожалуйста, обратитесь к соответствующим специалистам.'
    )}


# ── Маршрутизатор ─────────────────────────────────────────────
def route_after_classify(state: AssistantState) -> str:
    return 'answer' if state['is_hotel_related'] else 'out_of_scope'


# ── Сборка графа ───────────────────────────────────────────────
ab = StateGraph(AssistantState)
ab.add_node('classify',     classify_question)
ab.add_node('answer',       answer_question)
ab.add_node('out_of_scope', out_of_scope_response)
ab.add_edge(START, 'classify')
ab.add_conditional_edges('classify', route_after_classify,
                         {'answer': 'answer', 'out_of_scope': 'out_of_scope'})
ab.add_edge('answer',       END)
ab.add_edge('out_of_scope', END)
assistant_graph = ab.compile()

print('✅ Граф-ассистент собран')

✅ Граф-ассистент собран


In [35]:
# ── Быстрая проверка ──────────────────────────────────────────
BLANK = {'question': '', 'is_hotel_related': None, 'answer': ''}

print('=' * 65)
print('Тест 1: вопрос по теме отеля')
r1 = assistant_graph.invoke({**BLANK, 'question': 'До какого часа работает бассейн?'})
print(f'  По теме: {r1["is_hotel_related"]}')
print(f'  Ответ:   {r1["answer"][:130]}')

print()
print('Тест 2: out-of-scope вопрос')
r2 = assistant_graph.invoke({**BLANK, 'question': 'Какой сейчас курс доллара к евро?'})
print(f'  По теме: {r2["is_hotel_related"]}')
print(f'  Ответ:   {r2["answer"]}')
print('=' * 65)

Тест 1: вопрос по теме отеля
  По теме: True
  Ответ:   Бассейн работает ежедневно с 6:00 до 22:00, при условии подходящей погоды.

Тест 2: out-of-scope вопрос
  По теме: False
  Ответ:   Извините, я специализируюсь только на вопросах, связанных с The Grand Harbor Hotel — заселение, услуги, удобства, правила и т.п. По другим темам, пожалуйста, обратитесь к соответствующим специалистам.


## 7. Граф-пайплайн оценки (LangGraph)

```
START
  │
  ▼
[process_item]  ← вопрос[idx] → ассистент → LLM-Judge + ANLS
  │
  ├── ещё вопросы? ──► [process_item]       (цикл)
  └── всё готово?  ──► [aggregate_results] → END
```

In [36]:
# ── Состояние пайплайна ────────────────────────────────────────
class EvalState(TypedDict):
    qa_pairs:    List[dict]
    current_idx: int
    results: Annotated[List[dict], operator.add]  # auto-append


# ── Узел: обработка одного вопроса ────────────────────────────
def process_item(state: EvalState) -> dict:
    idx = state['current_idx']
    qa  = state['qa_pairs'][idx]

    print(f'  [{idx+1:>2}/20] {qa["question"][:60]}')

    # 1. Получить ответ от ассистента
    res    = assistant_graph.invoke({**BLANK, 'question': qa['question']})
    actual = res['answer']

    # 2. Вычислить метрики
    llm_s  = llm_judge_score(qa['question'], actual, qa['answer'])
    anls_s = compute_anls(actual, qa['answer'])

    print(f'         LLM-Judge={llm_s:.2f}  ANLS={anls_s:.2f}')

    return {
        'current_idx': idx + 1,
        'results': [{
            'id':         qa['id'],
            'question':   qa['question'],
            'expected':   qa['answer'],
            'actual':     actual,
            'llm_score':  llm_s,
            'anls_score': anls_s,
        }],
    }


# ── Условный маршрутизатор ─────────────────────────────────────
def should_continue(state: EvalState) -> str:
    return 'process' if state['current_idx'] < len(state['qa_pairs']) else 'done'


# ── Узел: агрегация ───────────────────────────────────────────
def aggregate_results(state: EvalState) -> dict:
    results  = state['results']
    avg_llm  = sum(r['llm_score']  for r in results) / len(results)
    avg_anls = sum(r['anls_score'] for r in results) / len(results)

    W = 92
    print()
    print('=' * W)
    print(' РЕЗУЛЬТАТЫ ОЦЕНКИ QA-АССИСТЕНТА (GigaChat) — The Grand Harbor Hotel')
    print('=' * W)
    print(f'{"ID":>3} | {"Вопрос":<52} | {"LLM-Judge":>9} | {"ANLS":>6}')
    print('-' * W)
    for r in results:
        q = (r['question'][:51] + '…') if len(r['question']) > 52 else r['question']
        print(f"{r['id']:>3} | {q:<52} | {r['llm_score']:>9.3f} | {r['anls_score']:>6.3f}")
    print('=' * W)
    print(f"{'ср':>3}   {'(20 вопросов)':<52}   {avg_llm:>9.3f}   {avg_anls:>6.3f}")
    print('=' * W)
    print(f'\n📊 Итог: LLM-as-a-Judge = {avg_llm:.3f}  |  ANLS = {avg_anls:.3f}')
    return {}


# ── Сборка пайплайна ───────────────────────────────────────────
eb = StateGraph(EvalState)
eb.add_node('process',   process_item)
eb.add_node('aggregate', aggregate_results)
eb.add_edge(START, 'process')
eb.add_conditional_edges('process', should_continue,
                         {'process': 'process', 'done': 'aggregate'})
eb.add_edge('aggregate', END)
eval_graph = eb.compile()

print('✅ Граф-пайплайн оценки собран')

✅ Граф-пайплайн оценки собран


## 8. Запуск пайплайна

In [25]:
print('🚀 Запуск оценочного пайплайна (GigaChat)...')
print('-' * 65)

final_state = eval_graph.invoke({
    'qa_pairs':    QA_PAIRS,
    'current_idx': 0,
    'results':     [],
})

print('\n✅ Пайплайн завершён')

🚀 Запуск оценочного пайплайна (GigaChat)...
-----------------------------------------------------------------
  [ 1/20] Сколько гостей можно разместить в номере?
         LLM-Judge=0.80  ANLS=0.00
  [ 2/20] Можно ли с домашними животными?
         LLM-Judge=0.90  ANLS=0.00
  [ 3/20] Можно ли курить в номерах?
         LLM-Judge=0.20  ANLS=0.00
  [ 4/20] Завтрак включён в стоимость?
         LLM-Judge=0.80  ANLS=0.00
  [ 5/20] Какие условия отмены брони?
         LLM-Judge=0.90  ANLS=0.70
  [ 6/20] Во сколько заезд и выезд?
         LLM-Judge=0.80  ANLS=0.59
  [ 7/20] Есть ли парковка?
         LLM-Judge=0.80  ANLS=0.55
  [ 8/20] Есть ли Wi-Fi и бесплатный ли он?
         LLM-Judge=0.80  ANLS=0.00
  [ 9/20] Что входит в номер?
         LLM-Judge=0.80  ANLS=0.00
  [10/20] Какие часы работы бассейна и фитнес-центра?
         LLM-Judge=0.80  ANLS=0.00
  [11/20] Можно ли заселиться после полуночи?
         LLM-Judge=0.80  ANLS=0.00
  [12/20] Можно ли запросить дополнительную кровать или дет

## 9. Детальный анализ и сравнение метрик (бонус)

In [26]:
results = final_state['results']

# ── Топ расхождений между метриками ───────────────────────────
by_div = sorted(results, key=lambda r: abs(r['llm_score'] - r['anls_score']), reverse=True)

print('📌 Топ-5 расхождений LLM-Judge vs ANLS:')
print('=' * 85)
for r in by_div[:5]:
    diff = r['llm_score'] - r['anls_score']
    sign = '+' if diff > 0 else ''
    print(f"\n[Q{r['id']}] {r['question']}")
    print(f"  LLM-Judge={r['llm_score']:.3f}  ANLS={r['anls_score']:.3f}  Δ={sign}{diff:.3f}")
    print(f"  Эталон: {r['expected']}")
    print(f"  Ответ:  {r['actual']}")

📌 Топ-5 расхождений LLM-Judge vs ANLS:

[Q2] Можно ли с домашними животными?
  LLM-Judge=0.900  ANLS=0.000  Δ=+0.900
  Эталон: Да, мы pet-friendly отель! Принимаем собак и кошек, доплата за животное — $25 за ночь. В общих зонах животные должны быть на поводке, оставлять их одних в номере нельзя. По запросу предоставляем лежанки и миски. Пожалуйста, сообщите о животном при бронировании.
  Ответ:  Да, мы принимаем домашних животных – собак и кошек ($25/ночь). Животных необходимо держать на поводке в общих зонах, и нельзя оставлять их одних в номере. Лежанки и миски можно заказать заранее. Укажите это при бронировании.

[Q12] Можно ли запросить дополнительную кровать или детскую кроватку?
  LLM-Judge=0.900  ANLS=0.000  Δ=+0.900
  Эталон: Да, раскладные кровати — $25 за ночь, их количество ограничено, поэтому запрашивайте заранее. Детские кроватки предоставляются бесплатно по запросу. Также бесплатно можно получить манеж. Позвоните минимум за 24 часа до заезда.
  Ответ:  Да, можно запросит

In [27]:
# ── Худшие ответы по LLM-Judge ────────────────────────────────
worst3 = sorted(results, key=lambda r: r['llm_score'])[:3]

print('⚠️  Вопросы с наименьшим LLM-Judge score:')
print('=' * 85)
for r in worst3:
    print(f"\n[Q{r['id']}] {r['question']}")
    print(f"  LLM-Judge={r['llm_score']:.3f}  ANLS={r['anls_score']:.3f}")
    print(f"  Эталон: {r['expected'][:120]}")
    print(f"  Ответ:  {r['actual'][:120]}")

⚠️  Вопросы с наименьшим LLM-Judge score:

[Q3] Можно ли курить в номерах?
  LLM-Judge=0.200  ANLS=0.000
  Эталон: Все наши номера — для некурящих. Места для курения оборудованы на открытой террасе на 3-м этаже и во внутреннем дворе. З
  Ответ:  Извините, я специализируюсь только на вопросах, связанных с The Grand Harbor Hotel — заселение, услуги, удобства, правил

[Q1] Сколько гостей можно разместить в номере?
  LLM-Judge=0.800  ANLS=0.000
  Эталон: Стандартные номера рассчитаны на 2 взрослых. В номерах категории Deluxe могут разместиться до 4 гостей. Дети до 12 лет п
  Ответ:  В стандартном номере – до 2 взрослых, в номере Deluxe – до 4 гостей, в сюите – до 6 гостей.

[Q4] Завтрак включён в стоимость?
  LLM-Judge=0.800  ANLS=0.000
  Эталон: Включён ли завтрак, зависит от тарифа. Тарифы 'С завтраком' включают завтрак с 6:30 до 10:00 ежедневно. Если завтрак не 
  Ответ:  Нет, завтрак не включён в стоимость. Он доступен на отдельных тарифах и стоит $15/чел.


In [28]:
# ── Финальная сводка ──────────────────────────────────────────
avg_llm  = sum(r['llm_score']  for r in results) / len(results)
avg_anls = sum(r['anls_score'] for r in results) / len(results)
best_r   = max(results, key=lambda r: r['llm_score'])
worst_r  = min(results, key=lambda r: r['llm_score'])

print('=' * 62)
print('📊 ФИНАЛЬНАЯ СВОДКА — GigaChat QA-ассистент')
print('=' * 62)
print(f'  Вопросов оценено:        {len(results)}')
print(f'  Средний LLM-as-a-Judge:  {avg_llm:.3f}  ({avg_llm*100:.1f}/100)')
print(f'  Средний ANLS:            {avg_anls:.3f}  ({avg_anls*100:.1f}/100)')
print(f"  Лучший ответ (LLM):      Q{best_r['id']} = {best_r['llm_score']:.3f}")
print(f"  Худший ответ (LLM):      Q{worst_r['id']} = {worst_r['llm_score']:.3f}")
print('=' * 62)

print()
print('🔍 Out-of-scope тест:')
oos = assistant_graph.invoke({**BLANK, 'question': 'Посоветуй лучший ресторан в городе'})
print(f'  Q: Посоветуй лучший ресторан в городе')
print(f"  По теме отеля: {oos['is_hotel_related']}")
print(f"  A: {oos['answer'][:200]}")

📊 ФИНАЛЬНАЯ СВОДКА — GigaChat QA-ассистент
  Вопросов оценено:        20
  Средний LLM-as-a-Judge:  0.795  (79.5/100)
  Средний ANLS:            0.216  (21.6/100)
  Лучший ответ (LLM):      Q2 = 0.900
  Худший ответ (LLM):      Q3 = 0.200

🔍 Out-of-scope тест:
  Q: Посоветуй лучший ресторан в городе
  По теме отеля: False
  A: Извините, я специализируюсь только на вопросах, связанных с The Grand Harbor Hotel — заселение, услуги, удобства, правила и т.п. По другим темам, пожалуйста, обратитесь к соответствующим специалистам.


## Когда метрики расходятся?

**LLM-Judge > ANLS — типичные случаи:**
- Ответ верен по смыслу, но перефразирован: «с шести утра» vs «6:00». ANLS видит разные строки, LLM понимает совпадение.
- Ответ подробнее эталона — добавлены полезные уточнения. ANLS штрафует за длину, LLM оценивает полноту как плюс.
- Порядок перечисления отличается от эталона.

Кейсов **LLM-Judge < ANLS** вообще нет. 

**Итог:** ANLS — быстрый индикатор, удобный при разработке и дебаггинге.  
LLM-as-a-Judge — точный арбитр для production-оценки. Вместе они покрывают слепые зоны друг друга.